# DistilBERT Phishing Classifier — Training Notebook

**Sentinel Zero Local** — ML Model Training

This notebook trains a DistilBERT-based binary classifier to detect phishing URLs.
The model is fine-tuned on:
- **PhishTank dataset**: 50,000 verified phishing URLs
- **Indian phishing corpus**: 5,000 India-specific phishing samples

After training, the model is exported to ONNX format for browser-compatible inference.

## Requirements
- GPU recommended (≥8GB VRAM) — ~45min on RTX 3070, ~8hr on CPU
- RAM: ≥16GB recommended

## 1. Install Dependencies

In [ ]:
# TODO: Run this cell first to install required packages
# !pip install transformers torch onnxruntime onnx datasets scikit-learn pandas tqdm


## 2. Imports & Configuration

In [ ]:
import os
import sys
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    AdamW,
    get_linear_schedule_with_warmup,
)
from tqdm.auto import tqdm

# Configuration
MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 3
LR = 2e-5
WARMUP_STEPS = 500
RANDOM_SEED = 42
OUTPUT_DIR = Path('../models')
OUTPUT_DIR.mkdir(exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')


## 3. Load and Prepare Dataset

In [ ]:
# Load PhishTank phishing URLs
# TODO: Download from https://data.phishtank.com/data/online-valid.json.bz2
# and save as ../data/phishing_urls.csv
phishing_df = pd.read_csv('../data/phishing_urls.csv')
phishing_df['label'] = 1
print(f'PhishTank samples: {len(phishing_df)}')

# Load Indian phishing corpus
indian_df = pd.read_csv('../data/indian_phishing_samples.csv')
indian_df = indian_df[['url']].copy()
indian_df['label'] = 1
print(f'Indian corpus samples: {len(indian_df)}')

# Load legitimate URLs
# TODO: Download Tranco list from https://tranco-list.eu/
legit_df = pd.read_csv('../data/legitimate_urls.csv')
legit_df['label'] = 0
print(f'Legitimate URL samples: {len(legit_df)}')

# Combine and balance
all_phishing = pd.concat([phishing_df[['url', 'label']], indian_df[['url', 'label']]])
all_legit = legit_df[['url', 'label']].sample(
    min(len(all_phishing), len(legit_df)), random_state=RANDOM_SEED
)

combined = pd.concat([all_phishing, all_legit]).sample(frac=1, random_state=RANDOM_SEED)
combined = combined.dropna(subset=['url'])
print(f'\nTotal samples: {len(combined)}')
print(combined['label'].value_counts())


## 4. Train/Validation/Test Split

In [ ]:
# 80% train, 10% val, 10% test
train_df, temp_df = train_test_split(
    combined, test_size=0.2, stratify=combined['label'], random_state=RANDOM_SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label'], random_state=RANDOM_SEED
)

print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')


## 5. Dataset Class

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

class PhishingDataset(Dataset):
    def __init__(self, urls, labels, tokenizer, max_len):
        self.urls = urls
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.urls)

    def __getitem__(self, idx):
        url = str(self.urls[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            url,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(label, dtype=torch.long),
        }

train_dataset = PhishingDataset(
    train_df['url'].values, train_df['label'].values, tokenizer, MAX_LEN
)
val_dataset = PhishingDataset(
    val_df['url'].values, val_df['label'].values, tokenizer, MAX_LEN
)
test_dataset = PhishingDataset(
    test_df['url'].values, test_df['label'].values, tokenizer, MAX_LEN
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f'Batches — Train: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}')


## 6. Model Setup

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)
model = model.to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps,
)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Training steps: {total_steps}')


## 7. Training Loop

In [ ]:
def evaluate(model, loader):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['label'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()

            preds = outputs.logits.argmax(dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), np.array(all_preds), np.array(all_labels)


best_val_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0

    for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}'):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    val_loss, val_preds, val_labels = evaluate(model, val_loader)
    val_report = classification_report(
        val_labels, val_preds, target_names=['Legitimate', 'Phishing']
    )

    print(f'\nEpoch {epoch+1}: Train Loss={avg_train_loss:.4f}, Val Loss={val_loss:.4f}')
    print(val_report)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        model.save_pretrained(OUTPUT_DIR / 'distilbert_phishing_best')
        tokenizer.save_pretrained(OUTPUT_DIR / 'distilbert_phishing_best')
        print(f'  ✅ Saved best model (val_loss={val_loss:.4f})')


## 8. Test Set Evaluation

In [ ]:
# Load best model
best_model = DistilBertForSequenceClassification.from_pretrained(
    OUTPUT_DIR / 'distilbert_phishing_best'
).to(DEVICE)

_, test_preds, test_labels = evaluate(best_model, test_loader)

print('=== TEST SET RESULTS ===')
print(classification_report(test_labels, test_preds, target_names=['Legitimate', 'Phishing']))
print('\nConfusion Matrix:')
print(confusion_matrix(test_labels, test_preds))

# Calculate TPR and FPR explicitly
cm = confusion_matrix(test_labels, test_preds)
tn, fp, fn, tp = cm.ravel()
tpr = tp / (tp + fn)
fpr = fp / (fp + tn)
print(f'\nTPR (Recall): {tpr:.4f} ({tpr*100:.1f}%)')
print(f'FPR: {fpr:.4f} ({fpr*100:.1f}%)')

# Target: TPR >= 92%, FPR <= 3%
assert tpr >= 0.92, f'TPR {tpr:.3f} below target 0.92'
assert fpr <= 0.03, f'FPR {fpr:.3f} above target 0.03'
print('\n✅ All performance targets met!')


## 9. Export to ONNX

In [ ]:
import torch.onnx

best_model.eval()
onnx_path = OUTPUT_DIR / 'distilbert_phishing.onnx'

# Create dummy input for tracing
dummy_url = 'http://example.com'
dummy_enc = tokenizer(
    dummy_url,
    return_tensors='pt',
    max_length=MAX_LEN,
    padding='max_length',
    truncation=True,
)

torch.onnx.export(
    best_model,
    (
        dummy_enc['input_ids'].to(DEVICE),
        dummy_enc['attention_mask'].to(DEVICE),
    ),
    str(onnx_path),
    opset_version=12,
    input_names=['input_ids', 'attention_mask'],
    output_names=['logits'],
    dynamic_axes={
        'input_ids': {0: 'batch_size'},
        'attention_mask': {0: 'batch_size'},
        'logits': {0: 'batch_size'},
    },
)

print(f'ONNX model saved to: {onnx_path}')
print(f'ONNX model size: {onnx_path.stat().st_size / 1024 / 1024:.1f} MB')


## 10. Quantize ONNX Model

In [ ]:
# Optional: quantize to INT8 for smaller file size (~22MB vs ~260MB)
# !pip install onnxruntime-tools

from onnxruntime.quantization import quantize_dynamic, QuantType

quantized_path = OUTPUT_DIR / 'distilbert_phishing_int8.onnx'
quantize_dynamic(
    str(onnx_path),
    str(quantized_path),
    weight_type=QuantType.QInt8,
)

print(f'Quantized model: {quantized_path}')
print(f'Original size: {onnx_path.stat().st_size / 1024 / 1024:.1f} MB')
print(f'Quantized size: {quantized_path.stat().st_size / 1024 / 1024:.1f} MB')


## 11. Verify ONNX Inference

Verify the exported model produces the same outputs as the PyTorch model.

In [ ]:
import onnxruntime as ort
import time

session = ort.InferenceSession(str(quantized_path))

test_urls = [
    'http://paypal-secure-login.xyz/account/verify?token=abc123',
    'https://www.google.com/search?q=python',
    'http://uidai-aadhaar-update.xyz/verify',
    'https://github.com/Pranavyadav9519',
    'http://sbi-netbanking-secure.top/signin',
]

print('ONNX Inference Results:')
print('-' * 60)

for url in test_urls:
    enc = tokenizer(
        url,
        return_tensors='np',
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
    )

    start = time.perf_counter()
    logits = session.run(
        ['logits'],
        {
            'input_ids': enc['input_ids'].astype(np.int64),
            'attention_mask': enc['attention_mask'].astype(np.int64),
        },
    )[0]
    latency_ms = (time.perf_counter() - start) * 1000

    proba = np.exp(logits) / np.exp(logits).sum(axis=-1, keepdims=True)
    phishing_prob = float(proba[0][1])
    verdict = 'PHISHING' if phishing_prob > 0.5 else 'SAFE'

    print(f'{verdict:10s} ({phishing_prob:.3f}) [{latency_ms:.0f}ms] {url[:60]}')

print(f'\n✅ ONNX model working correctly!')
